In [1]:
import re, time, requests
from pathlib import Path
from bs4 import BeautifulSoup
import re
import pandas as pd
from datasets import load_dataset

/Users/syeramaddu/Coding/optimalStopping/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fomc = load_dataset("Coding-Fish/fomc-statements", split="train").to_pandas()
fomc.columns = fomc.columns.str.lower()            # Date/Text/Chair -> date/text/chair
fomc["date"] = pd.to_datetime(fomc["date"])
fomc = fomc[fomc["date"].between("2010-01-01", "2024-12-31")]

print(fomc.groupby(fomc["date"].dt.year).size())
print(fomc[fomc["date"].between("2020-03-01", "2020-03-31")]["date"])

date
2010     8
2011     8
2012     8
2013     8
2014     8
2015     8
2016     8
2017     8
2018     8
2019     8
2020    12
2021     8
2022     8
2023     8
2024     8
dtype: int64
161   2020-03-03
162   2020-03-15
163   2020-03-23
164   2020-03-31
Name: date, dtype: datetime64[us]


In [ ]:
fomc = fomc[~fomc["date"].isin(pd.to_datetime(["2020-03-31", "2020-08-27"]))].copy()

fomc["release_time"] = fomc["text"].str.extract(r"For release at ([\d:]+ [ap]\.m\. E[SD]T)")[0]
fomc["has_vote"] = fomc["text"].str.contains("Voting for")

BOILER = re.compile(
    r"^(share|press release|(federal reserve issues )?fomc statement|for (immediate )?release.*"
    r"|release date:.*|[a-z]+ \d{1,2}, \d{4})$", re.I)

def clean_fomc(t):
    t = t.split("Voting for", 1)[0]                        # drop the vote list and everything after
    t = t.split("For media inquiries", 1)[0]
    lines = [ln.strip() for ln in t.splitlines()]
    return "\n".join(ln for ln in lines if ln and not BOILER.match(ln))

fomc["text"] = fomc["text"].apply(clean_fomc)

print(fomc.groupby(fomc["date"].dt.year).size())
print("missing vote line:", fomc.loc[~fomc["has_vote"], "date"].dt.date.tolist())

date
2010     8
2011     8
2012     8
2013     8
2014     8
2015     8
2016     8
2017     8
2018     8
2019     8
2020    10
2021     8
2022     8
2023     8
2024     8
dtype: int64
missing vote line: [datetime.date(2020, 3, 23)]


In [4]:
first = fomc.groupby(fomc["date"].dt.year).head(1)
for d, t in zip(first["date"], first["text"]):
    print(d.date(), "|", t[:150].replace("\n", " "))

2010-01-27 | Information received since the Federal Open Market Committee met in December suggests that economic activity has continued to strengthen and that the 
2011-01-26 | Information received since the Federal Open Market Committee met in December confirms that the economic recovery is continuing, though at a rate that 
2012-01-25 | Information received since the Federal Open Market Committee met in December suggests that the economy has been expanding moderately, notwithstanding 
2013-01-30 | Information received since the Federal Open Market Committee met in December suggests that growth in economic activity paused in recent months, in lar
2014-01-29 | Information received since the Federal Open Market Committee met in December indicates that growth in economic activity picked up in recent quarters. 
2015-01-28 | Information received since the Federal Open Market Committee met in December suggests that economic activity has been expanding at a solid pace.  Labo
2016-01-27 | Inf

In [5]:
BASE = "https://www.ecb.europa.eu"
HEADERS = {"User-Agent": "Mozilla/5.0 (academic research)"}
CACHE = Path("data/raw/ecb")
CACHE.mkdir(parents=True, exist_ok=True)

def get(url, retries=3):
    """Download a page once (retrying on network errors), then reuse the saved copy."""
    path = CACHE / url.replace(BASE, "").strip("/").replace("/", "_")
    if not path.exists():
        for attempt in range(retries):
            try:
                r = requests.get(url, headers=HEADERS, timeout=60)
                r.raise_for_status()
                break
            except requests.exceptions.RequestException as e:
                if attempt == retries - 1:
                    raise                              # give up after the last try
                wait = 5 * 2**attempt                  # 5s, then 10s
                print(f"retry {attempt + 1} for {url.split('/')[-1]} in {wait}s ({type(e).__name__})")
                time.sleep(wait)
        path.write_text(r.text, encoding="utf-8")
        time.sleep(2)                                  # slower pace between pages
    return path.read_text(encoding="utf-8")

In [6]:
links = []
for year in range(2010, 2025):
    page = get(f"{BASE}/press/press_conference/monetary-policy-statement/{year}/html/index_include.en.html")
    for a in BeautifulSoup(page, "html.parser").find_all("a", href=True):
        if re.search(r"/(?:ecb\.)?is\d{6}[^/]*\.en\.html$", a["href"]):
            links.append(a["href"] if a["href"].startswith("http") else BASE + a["href"])

links = sorted(set(links))
dates = pd.to_datetime([re.search(r"is(\d{6})", u).group(1) for u in links], format="%y%m%d")
print(pd.Series(dates.year).value_counts().sort_index())

2010    12
2011    12
2012    12
2013    12
2014    13
2015     8
2016     8
2017     8
2018     8
2019     8
2020     8
2021     8
2022     8
2023     8
2024     8
Name: count, dtype: int64


In [7]:
END  = re.compile(r"We are now (?:at your disposal for|ready to take your) questions", re.I)
SEP  = re.compile(r"\*\s*\*\s*\*")
JUMP = "Jump to the transcript of the questions and answers"

rows = []
for url, date in zip(links, dates):
    soup = BeautifulSoup(get(url), "html.parser")
    full = (soup.find("main") or soup).get_text("\n", strip=True)

    m = END.search(full) or SEP.search(full)          # specific phrase first, divider as fallback
    stmt = full[:m.start()] if m else full
    if JUMP in stmt:
        stmt = stmt.split(JUMP, 1)[1]                 # drop page header
    stmt = stmt.split("the floor is yours", 1)[-1]    # drop host welcome speeches (Berlin 2011)

    rows.append({"date": date, "url": url, "cut_found": m is not None, "text": stmt.strip()})

ecb = pd.DataFrame(rows)
ecb = ecb[ecb["date"] != "2014-10-26"]                # stress-test press conference, not monetary policy

print(ecb["cut_found"].value_counts())
print(ecb.groupby(ecb["date"].dt.year).size())
ecb[ecb["text"].str.len() < 5000][["date", "url"]]

cut_found
True    140
Name: count, dtype: int64
date
2010    12
2011    12
2012    12
2013    12
2014    12
2015     8
2016     8
2017     8
2018     8
2019     8
2020     8
2021     8
2022     8
2023     8
2024     8
dtype: int64


,date,url


In [8]:
ecb.assign(bank="ECB")[["bank", "date", "text"]]

,bank,date,text
0,ECB,2010-01-14,"Ladies and gentlemen, the Vice-President and I..."
1,ECB,2010-02-04,"Ladies and gentlemen, the Vice-President and I..."
2,ECB,2010-03-04,"Ladies and gentlemen, the Vice-President and I..."
3,ECB,2010-04-08,"Ladies and gentlemen, the Vice-President and I..."
4,ECB,2010-05-06,"Ladies and gentlemen, the Vice-President and I..."
...,...,...,...
136,ECB,2024-06-06,"Good afternoon, the Vice-President and I welco..."
137,ECB,2024-07-18,"Good afternoon, the Vice-President and I welco..."
138,ECB,2024-09-12,"Good afternoon, the Vice-President and I welco..."
139,ECB,2024-10-17,"Good afternoon, the Vice-President and I welco..."


In [9]:
statements = pd.concat([
    fomc.assign(bank="FOMC")[["bank", "date", "release_time", "text"]],
    ecb.assign(bank="ECB", release_time=None)[["bank", "date", "release_time", "text"]]
]).sort_values("date").reset_index(drop=True)

statements.to_parquet("data/statements.parquet")
statements.groupby("bank")["text"].apply(lambda s: s.str.len().median())

bank
ECB     9002.5
FOMC    2757.0
Name: text, dtype: float64

In [10]:
len(list(CACHE.glob("*")))

156

In [11]:
# 1. 2014: which one is the extra?
print(ecb[ecb["date"].dt.year == 2014][["date", "url"]].to_string())

# 2. Shortest statement: does it end mid-speech?
short = ecb.loc[ecb["text"].str.len().idxmin()]
print("\n", short["date"].date(), short["url"])
print(short["text"][-500:])

# 3. FOMC 2020: are all 12 policy statements?
print(fomc[fomc["date"].dt.year == 2020][["date", "text"]]
      .assign(text=lambda d: d["text"].str[:120]).to_string())

s = ecb[ecb["date"] == "2012-07-05"]["text"].iloc[0]
print(len(s)); print(s[:300])

         date                                                                                                    url
48 2014-01-09  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140109.en.html
49 2014-02-06  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140206.en.html
50 2014-03-06  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140306.en.html
51 2014-04-03  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140403.en.html
52 2014-05-08  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140508.en.html
53 2014-06-05  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140605.en.html
54 2014-07-03  https://www.ecb.europa.eu/press/press_conference/monetary-policy-statement/2014/html/is140703.en.html
55 2014-08-07  https://www.ecb.europa.eu/press/press_conference/